In [1]:
# =========================================
# Import packages and set up Neo4j
# =========================================
from dotenv import load_dotenv
import os

# Common data processing
import textwrap

# Langchain (modern)
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LCEL
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Warning control
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load env
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

In [3]:
# Constants
VECTOR_INDEX_NAME = 'form_10k_chunks'
VECTOR_NODE_LABEL = 'Chunk'
VECTOR_SOURCE_PROPERTY = 'text'
VECTOR_EMBEDDING_PROPERTY = 'textEmbedding'

# Models
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(temperature=0)

In [4]:
# Neo4j
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [5]:
# =========================================
# Create a Form 10-K node
# =========================================
cypher = """
MATCH (c:Chunk)
WITH c LIMIT 1
RETURN c { .names, .source, .formId, .cik, .cusip6 } AS formInfo
"""

form_info = kg.query(cypher)[0]["formInfo"]

cypher = """
MERGE (f:Form {formId: $formInfoParam.formId})
ON CREATE SET 
    f.names = $formInfoParam.names,
    f.source = $formInfoParam.source,
    f.cik = $formInfoParam.cik,
    f.cusip6 = $formInfoParam.cusip6
"""

kg.query(cypher, params={'formInfoParam': form_info})

print(kg.query("MATCH (f:Form) RETURN count(f) AS formCount"))

[{'formCount': 1}]


In [6]:
# =========================================
# Create a linked list of Chunk nodes for each section
# =========================================
cypher = """
MATCH (c:Chunk)
WHERE c.formId = $formIdParam
  AND c.f10kItem = $f10kItemParam
WITH c ORDER BY c.chunkSeqId ASC
WITH collect(c) AS chunks
CALL apoc.nodes.link(chunks, "NEXT", {avoidDuplicates: true})
RETURN size(chunks)
"""

for item in ['item1', 'item1a', 'item7', 'item7a']:
    kg.query(cypher, params={
        "formIdParam": form_info["formId"],
        "f10kItemParam": item
    })

In [7]:
# =========================================
# Connect graph structure
# =========================================

# PART_OF relationship
kg.query("""
MATCH (c:Chunk), (f:Form)
WHERE c.formId = f.formId
MERGE (c)-[:PART_OF]->(f)
""")

# SECTION relationship
kg.query("""
MATCH (c:Chunk), (f:Form)
WHERE c.formId = f.formId AND c.chunkSeqId = 0
MERGE (f)-[:SECTION {f10kItem: c.f10kItem}]->(c)
""")


[]

In [9]:
# =========================================
# Example cypher queries
# =========================================

# First chunk
first_chunk = kg.query("""
MATCH (f:Form)-[r:SECTION]->(c:Chunk)
WHERE f.formId = $formIdParam AND r.f10kItem = $item
RETURN c.chunkId AS chunkId, c.text AS text
""", params={
    "formIdParam": form_info["formId"],
    "item": "item1"
})[0]

# Next chunk
next_chunk = kg.query("""
MATCH (c:Chunk)-[:NEXT]->(next:Chunk)
WHERE c.chunkId = $chunkId
RETURN next.chunkId AS chunkId, next.text AS text
""", params={"chunkId": first_chunk["chunkId"]})[0]

print(first_chunk["chunkId"], next_chunk["chunkId"])

0000950170-23-027948-item1-chunk0000 0000950170-23-027948-item1-chunk0001


In [10]:
# =========================================
# Graph traversal examples
# =========================================

kg.query("""
MATCH (c1:Chunk)-[:NEXT]->(c2:Chunk)-[:NEXT]->(c3:Chunk)
WHERE c2.chunkId = $chunkId
RETURN c1.chunkId, c2.chunkId, c3.chunkId
""", params={"chunkId": next_chunk["chunkId"]})

kg.query("""
MATCH window=(c1:Chunk)-[:NEXT]->(c2:Chunk)-[:NEXT]->(c3:Chunk)
WHERE c1.chunkId = $chunkId
RETURN length(window)
""", params={"chunkId": next_chunk["chunkId"]})

[{'length(window)': 2}]

In [11]:
# =========================================
# Vector retrieval customization
# =========================================

retrieval_query_extra_text = """
WITH node, score, "Andreas knows Cypher." AS extraText
RETURN extraText + "\n" + node.text AS text,
       score,
       node {.source} AS metadata
"""

vector_store_extra = Neo4jVector.from_existing_index(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    text_node_property=VECTOR_SOURCE_PROPERTY,
    retrieval_query=retrieval_query_extra_text,
)

retriever_extra = vector_store_extra.as_retriever()

In [12]:
# =========================================
# Modern RAG (extra text)
# =========================================

prompt = ChatPromptTemplate.from_template("""
Answer based only on the context:

{context}

Question:
{question}
""")

rag_extra = (
    {"context": retriever_extra, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_extra.invoke("What topics does Andreas know about?"))


Andreas knows about workplace safety, environmental regulations, compliance with environmental laws, ISO 14001 requirements, human capital, diversity, inclusion, belonging, benefits, wellbeing, engagement, healthcare options, insurance, and income protection.


In [13]:
# =========================================
# Window-based retrieval (Graph-aware RAG)
# =========================================

retrieval_query_window = """
MATCH window=
    (:Chunk)-[:NEXT*0..1]->(node)-[:NEXT*0..1]->(:Chunk)
WITH node, score, window
ORDER BY length(window) DESC LIMIT 1
WITH nodes(window) AS chunks, node, score
UNWIND chunks AS c
WITH collect(c.text) AS texts, node, score
RETURN apoc.text.join(texts, " \n ") AS text,
       score,
       node {.source} AS metadata
"""

vector_store_window = Neo4jVector.from_existing_index(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    text_node_property=VECTOR_SOURCE_PROPERTY,
    retrieval_query=retrieval_query_window,
)

retriever_window = vector_store_window.as_retriever()

rag_window = (
    {"context": retriever_window, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [14]:
# =========================================
# Compare outputs
# =========================================

question = "In a single sentence, tell me about Netapp's business."

print("\n--- Without window ---")
print(textwrap.fill(rag_extra.invoke(question), 80))

print("\n--- With window ---")
print(textwrap.fill(rag_window.invoke(question), 80))


--- Without window ---
NetApp is a global cloud-led, data-centric software company that provides
customers with the freedom to manage applications and data across hybrid
multicloud environments, offering operational simplicity, flexibility,
consistency, cyber resilience, continuous operations, and sustainability through
their portfolio of cloud services and storage infrastructure powered by
intelligent data management software.

--- With window ---
NetApp offers storage-as-a-service solutions, global support, and a multichannel
distribution strategy to cater to a diverse customer base in various industries,
focusing on enterprise storage, data management, cloud storage, and cloud
operations markets.
